# 投机解码：一次 Target Forward 能不能拿多个 Token？

> 量化降低了每次前向的存储和带宽成本，却没有改变自回归生成“一步一步来”的串行依赖。
>
> 投机解码（Speculative Decoding）从这里下手：让便宜的 Draft 先猜多个 Token，再让 Target 一次并行验证。
>
> 读完后，你应该能看懂 `draft model`、`acceptance rate`、`speculative sampling`、`Medusa` / multi-token prediction 等报告里反复出现的核心思想。


## 1. 串行瓶颈

```text
普通 Decode:
Target -> token1
Target -> token2
Target -> token3
Target -> token4
```

即使 KV Cache 已经存在，四个 Token 仍需要四次串行 Target step。

投机解码希望变成：

```text
Draft 快速猜: token1 token2 token3 token4
                ↓
Target 一次验证多个位置
                ↓
接受一段前缀 + 必要时修正
```


## 2. Draft / Verify / Accept

最经典的流程可以拆成三步：

1. Draft model 连续生成 K 个候选
2. Target model 对这些候选位置并行得到概率
3. 按接受规则保留前缀；第一个拒绝处用校正分布重新采样

真正的难点不是“猜”，而是：**快了以后，输出分布还能和原 Target sampling 一样吗？**


In [ ]:
draft_probs  = [0.80, 0.70, 0.60, 0.50]
target_probs = [0.90, 0.80, 0.20, 0.10]

for i,(q,p) in enumerate(zip(draft_probs,target_probs)):
    accept = min(1.0, p/q)
    print(f"pos {i}: target/draft={p/q:.2f}, accept_prob={accept:.2f}")


## 3. 为什么接受概率是 `min(1, p/q)`？

如果 Draft 对某个 Token 的概率是 `q(x)`，Target 是 `p(x)`，经典 speculative sampling 的接受概率是：

$$
a(x)=\min(1, p(x)/q(x))
$$

但只写这个公式还不够。**拒绝之后还需要从校正分布采样**，这样整个算法才保持 Target 原来的输出分布。


## 4. 速度由什么决定？

粗略看三件事：

- Draft 有多便宜
- 每轮猜几个 Token（K）
- Acceptance Rate 有多高

K 不是越大越好。猜太多但接受率低，Draft 白算；Draft 太大，也可能把省下来的 Target 成本吃掉。


In [ ]:
def toy_speedup(k, accept_rate, draft_cost_ratio):
    # 一个教学用 proxy：每轮期望拿到 1 + 接受的 draft tokens
    expected_tokens = 1 + k * accept_rate
    cost = 1 + k * draft_cost_ratio
    return expected_tokens / cost

for a in [0.3,0.6,0.9]:
    print("accept", a, "proxy speedup", round(toy_speedup(4,a,0.08),2))


## 5. 最重要的实验：快了，但分布不能变

下面用一个两 Token 的离散分布做 Monte Carlo。

Target 真实分布是 `[0.7, 0.3]`。我们故意让 Draft 分布不同，然后执行一轮 speculative sampling 的接受 + 校正。大量采样后，最终频率应该重新回到 Target。


In [ ]:
import random, collections

p = [0.7, 0.3]  # target
q = [0.5, 0.5]  # draft

def sample(dist):
    r = random.random()
    return 0 if r < dist[0] else 1

def speculative_one():
    x = sample(q)
    if random.random() < min(1.0, p[x] / q[x]):
        return x
    residual = [max(p[i]-q[i],0.0) for i in range(2)]
    s = sum(residual)
    if s == 0:
        return sample(p)
    residual = [v/s for v in residual]
    return sample(residual)

random.seed(42)
n=50000
c=collections.Counter(speculative_one() for _ in range(n))
print("target:", p)
print("speculative empirical:", [round(c[i]/n,3) for i in range(2)])


## 6. 厂商报告里还会看到哪些“同一家族”名词？

| 名词 | 核心想法 |
|---|---|
| Speculative Decoding | Draft + Target verify |
| Speculative Sampling | 保持 sampling 分布的完整接受 / 校正规则 |
| Self-speculative | 不额外放一个独立 Draft，利用同模型浅层 / 子网络等方式 |
| Medusa / EAGLE 等 | 用额外 head / 特征预测一次提出多个未来 Token |
| Multi-token prediction | 模型训练或头部直接预测多个未来位置 |

实现不同，但你可以先问同一个问题：

> **谁在提案？谁在验证？一次 Target 计算最终平均确认几个 Token？**


下一章视角再升一层：

> **单个请求已经优化很多，但如果 100 个请求同时进一张 GPU，谁先算、KV Cache 放哪、长 Prompt 和 Decode 怎么抢资源？**
